# Case study 2: One-Tap Apply v2, a cluster-randomized test

*Sequel to Case 1. Synthetic data, modeled on how two-sided job marketplaces run experiments.*

## The brief

Your Case 1 memo said: don't ship One-Tap Apply as-is, keep required screening questions, and **randomize by job posting next time** so recruiter-side effects are measured cleanly. Leadership agreed.

**One-Tap v2:** members still apply in one tap, but required screening questions stay. Only optional questions are skipped.

**Design:** every job posting created between Nov 30 and Dec 6, 2026 was randomly assigned to treatment or control **when it was created**. Everyone who applies to a treatment posting sees One-Tap v2. Applications were tracked for 4 weeks.

The PM writes:

> "v2 is live on half of postings. Quick heads-up: the dashboard says recruiter response rate is significantly down again (p = 0.002). Should we kill it?"

**Your job:** figure out whether that's real, measure the effect on applications and quality correctly, and make a launch call.

## What's new in this case

The unit of randomization is now a **cluster** (a job posting) that contains many applications. Applications to the same posting share a recruiter, a company and a job, so their outcomes are correlated. That changes how you have to analyze every metric.

## Data (in `data/`)

| File | Grain | Columns |
|---|---|---|
| `postings.csv` | 1 row per job posting in the test | `posting_id`, `variant`, `created_date`, `job_function`, `seniority`, `company_size`, `company_avg_apps_pre` (avg applications per posting this company got in the 90 days **before** the test), `recruiter_resp_rate_pre` (this recruiter's response rate in the 90 days **before** the test) |
| `applications.csv` | 1 row per application | `application_id`, `posting_id`, `member_id`, `applied_date`, `recruiter_responded_7d`, `recruiter_marked_qualified` (recruiter tagged the applicant as a fit) |

## Methods you'll practice

| New | Review from Case 1 |
|---|---|
| Intraclass correlation (ICC) | SRM check |
| Design effect and effective sample size | Welch t-test |
| Cluster-level analysis | CUPED |
| Cluster bootstrap | Delta method for ratio metrics |
| Covariate balance (standardized mean difference) | Launch memo |
| Power for a cluster-randomized test | |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
DATA = "data/"
ALPHA = 0.05

---
# Part A: Design questions (answer before touching data)

**A1.** Randomizing by posting fixes the recruiter-side interference from Case 1. What new problem does it create on the **member** side? Which direction would it bias the applications result?

> Randomizing by posting fixes recruiter-side interference, but it risks a SUTVA violation on the member side. SUTVA assumes each unit's outcome depends only on its own assignment, so control postings should behave as if One-Tap didn't exist. Here, members aren't randomized: the same member sees both One-Tap and regular postings. A member who would have applied to a control posting may apply to an easier One-Tap posting instead. That shifts applications from control to treatment, inflating one and deflating the other, so the measured lift likely overstates the true effect. After a full launch, every posting has One-Tap, so there's nothing easier to shift to, and only genuinely new applications count.

**A2.** For each metric below, what is the unit of analysis, and is it a simple mean or a ratio metric?
- Applications per posting (primary)
- Recruiter response rate (guardrail)
- Qualified applicants per posting (employer-side "north star")

**A3.** The test has about 4,000 postings and over 100,000 applications. Which number is your sample size, and why?

**A4.** What would you use the two pre-period columns for?

**Your answer:**

_Write it here before looking at the answer key._

---
# Part B: Trust checks

## B1. Build a posting-level table

One row per posting from `postings.csv`, with:
- `apps`: number of applications
- `responses`: sum of `recruiter_responded_7d`
- `qualified`: sum of `recruiter_marked_qualified`

Postings with no applications must stay in, with zeros.

In [ ]:
postings = pd.read_csv(DATA + "postings.csv")
apps = pd.read_csv(DATA + "applications.csv")

# TODO: aggregate apps by posting_id, left-join onto postings, fill zeros
post = None  # replace

## B2. SRM on postings

Same chi-square check as Case 1, but count **postings** (the randomization unit), not applications.

**Question:** why would an SRM check on application counts be misleading here?

In [ ]:
# TODO: chi-square goodness of fit on posting counts by variant

**Checkpoint:** p should be well above 0.05.

## B3. Covariate balance

With fewer randomized units, chance imbalances are bigger. Check that the arms look alike **before** treatment using the **standardized mean difference (SMD)**:

SMD = (mean_T − mean_C) / √((var_T + var_C) / 2)

A common rule of thumb: |SMD| < 0.1 is balanced. Check the two pre-period numbers, plus the share of postings in each `company_size` and `seniority` category (use 0/1 dummies).

In [ ]:
def smd(df, col):
    # TODO
    raise NotImplementedError

# TODO: SMD for company_avg_apps_pre, recruiter_resp_rate_pre, and dummies of company_size and seniority

**Checkpoint:** every |SMD| should be below 0.1.

---
# Part C: The tempting (wrong) analysis

This is roughly what the PM's dashboard did. Treat every application as an independent observation and run a two-proportion z-test on:
1. Recruiter response rate
2. Qualified rate (qualified / applications)

In [ ]:
def naive_prop_test(df_apps, col):
    # TODO: application-level two-proportion z-test (pooled SE), return dict with rates and p-value
    raise NotImplementedError

# TODO: join variant onto apps, then run for recruiter_responded_7d and recruiter_marked_qualified

**Checkpoint:** the response rate should look significantly down (p ≈ 0.002), just like the PM said.

**Question:** before reading on, write down what assumption this test makes that's violated here.

---
# Part D: Measuring the clustering (ICC and design effect)

## D1. Intraclass correlation (ICC)

ICC is the share of the total variation in an outcome that comes from **differences between clusters**. ICC = 0 means postings don't matter (applications behave like independent coin flips). ICC = 1 means every application to the same posting gets the same outcome.

Estimate it with one-way ANOVA across postings (use all applications, both arms):

- MSB = between-posting mean square = Σ nᵢ (ȳᵢ − ȳ)² / (k − 1)
- MSW = within-posting mean square = Σ (y − ȳᵢ)² / (N − k)
- m₀ = (N − Σ nᵢ² / N) / (k − 1), an "average" cluster size adjusted for unequal sizes
- **ICC = (MSB − MSW) / (MSB + (m₀ − 1) · MSW)**

where k = number of postings with at least 1 application, nᵢ = applications to posting i, and N = total applications.

In [ ]:
def icc_anova(df, cluster, y):
    # TODO
    raise NotImplementedError

# TODO: ICC for recruiter_responded_7d and recruiter_marked_qualified

## D2. Design effect

The **design effect (DEFF)** is how much clustering inflates the variance compared to independent observations. With unequal cluster sizes, a good approximation is:

DEFF = 1 + (m_w − 1) · ICC, where m_w = Σ nᵢ² / Σ nᵢ (the size-weighted average cluster size)

Then:
- **Effective sample size** = N / DEFF
- **Correct SE ≈ naive SE × √DEFF**

Compute DEFF for response rate and apply it to the naive result from Part C. Is the drop still significant?

In [ ]:
# TODO: m_w from cluster sizes, DEFF, effective sample size
# TODO: inflate the naive SE for response rate, recompute z and p

**Checkpoint:** ICC for response rate should be around 0.1, but DEFF should be much bigger (around 9), because some postings get hundreds of applications.

**Question:** the ICC looks small. Why is the design effect still so big?

---
# Part E: The right way, two methods that should agree

## E1. Delta method at the posting level

Response rate = Σ responses / Σ applications is a **ratio metric**, and the posting is the randomization unit. That's exactly the Case 1 delta method setup, with posting in place of member. Reuse your `ratio_delta` function on the posting-level table.

In [ ]:
def ratio_delta(df, num, den):
    # TODO: (ratio, variance) for one arm
    raise NotImplementedError


def compare_ratio(df, num, den):
    # TODO: rates per arm, diff, SE, p-value
    raise NotImplementedError

# TODO: run for responses/apps and qualified/apps

## E2. Cluster bootstrap

No formulas: resample **postings** (never individual applications) with replacement within each arm, recompute each arm's rate, and repeat. The spread of the differences is the SE.

Write `cluster_bootstrap(df, num, den, B=2000)` that returns the bootstrap SE and a 95% percentile interval for the difference in rates.

**Hint (fast version):** for each arm, draw an index matrix `rng.integers(0, n, size=(B, n))`, then the rate for each replicate is `y[idx].sum(axis=1) / x[idx].sum(axis=1)`.

In [ ]:
def cluster_bootstrap(df, num, den, B=2000, seed=0):
    # TODO
    raise NotImplementedError

# TODO: run for both rates; compare SEs with naive (Part C), DEFF-adjusted (Part D) and delta (E1)

**Checkpoint:** delta and bootstrap SEs should be close to each other and about 3x the naive SE. The bootstrap CI for response rate should include 0.

**Question:** what's your verdict on the PM's "response rate is significantly down"?

## E3. Which average? (a judgment question)

Σ responses / Σ applications weights every **application** equally, so a few postings with hundreds of applicants dominate it. The alternative is the **average of each posting's own response rate**, which weights every **posting (employer)** equally.

Compute the posting-average response rate for each arm (postings with at least 1 application), with a Welch t-test. Which version would you report to the Jobs leadership, and why?

In [ ]:
# TODO: per-posting response rate, then compare means across arms

**Your answer:**

_Write it here before looking at the answer key._

---
# Part F: The primary metrics, done right

## F1. Applications per posting

Posting-level Welch t-test (the posting is both the randomization and analysis unit, so this is the easy case). Then add CUPED with `company_avg_apps_pre`. Reuse your Case 1 functions.

In [ ]:
def compare_means(df, col):
    # TODO (from Case 1)
    raise NotImplementedError


def cuped(df, y, x):
    # TODO (from Case 1)
    raise NotImplementedError

# TODO: raw and CUPED for apps; print rho and SE ratio

**Checkpoint:** raw lift ≈ +7% but not significant (p ≈ 0.10). CUPED should make it significant.

## F2. Qualified applicants per posting (the employer north star)

This is what employers actually want: more good applicants. Run raw and CUPED (same covariate).

In [ ]:
# TODO

**Checkpoint:** qualified applicants per posting should move up by a similar percentage to applications.

**Question:** how do F1, F2 and Part E fit together?

---
# Part G: Launch memo

About 150 words: decision, evidence, caveats, next steps. Include one sentence that explains the PM's false alarm in plain English.

**Your memo:**

_Decision:_

_Evidence:_

_Caveats:_

_Next steps:_

---
# Bonus: Interviewer follow-ups

1. **"Explain the design effect to a PM in one sentence."**
2. **"How would you measure the seeker-side cannibalization you mentioned?"**
3. **"For the next posting-level test, how many postings do you need to detect a 5% lift in applications per posting with 80% power?"** (Code it below: use the CV of `apps` per posting and the ρ from CUPED.)
4. **"Why not just randomize by recruiter or by company instead of by posting?"**

In [ ]:
# Stretch: n postings per arm = 2 (z_a + z_b)^2 * CV^2 * (1 - rho^2) / r^2
# TODO: compute with and without CUPED for r = 5%

_Your notes:_